### Pandas | Data Cleaning | Missing Values | GroupBy



---

## What You Will Learn

| # | Section | Skills |
|---|---------|--------|
| 0 | **Setup & Core Concepts** | Install, import, why Pandas exists |
| 1 | **Series & DataFrames** | Create, inspect, index, slice |
| 2 | **Selection & Filtering** | `[]`, `.loc`, `.iloc`, Boolean masks, `.query()` |
| 3 | **Data Cleaning** | Types, whitespace, duplicates, outliers, regex |
| 4 | **Missing Values** | Detect, drop, fill, interpolate, pipelines |
| 5 | **Transforming Data** | `apply`, `map`, `assign`, `cut`, `get_dummies` |
| 6 | **DateTime Operations** | Parse, resample, rolling windows |
| 7 | **Merging & Joining** | All join types, concat, merge_asof |
| 8 | **GroupBy & Aggregation** | split-apply-combine, pivot tables |
| 9 | **Performance & Best Practices** | dtypes, vectorize, memory, chaining |
| 10 | **Real-World Mini Project** | Full pipeline on a realistic dataset |



# Section 0 — Setup & Why Pandas?

---

## 0.1 Installation

Run this in a terminal (not needed in Google Colab — already installed):
```
pip install pandas numpy matplotlib seaborn
```


In [ ]:

# 0.2 — Imports used throughout this notebook
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Pandas  :", pd.__version__)
print("NumPy   :", np.__version__)
print("Ready!")


## 0.3 — Why Pandas?

**The problem Pandas solves:**  
Raw Python is bad at tables. Lists and dicts can hold data, but operations
like "give me all rows where salary > 50000" would need ugly loops.

Pandas gives you:
- A `DataFrame` (like an Excel sheet, but programmable)
- Blazing-fast vectorised math (no Python loops needed)
- Built-in handling for missing data, dates, text, categories
- Seamless I/O: CSV, Excel, JSON, SQL, Parquet, …

**Mental model:**
```
DataFrame  =  dictionary of Series
Series     =  NumPy array + index (row labels)
```


# Section 1 — Series & DataFrames

---
## 1.1 — The Pandas Series

A `Series` is a **labelled 1-D array**.  
Think of it as one column of a spreadsheet.


In [ ]:
# --- Create a Series from a list ---
temps = pd.Series([22, 25, 28, 24, 30, 27, 23])
print("Default integer index:")
print(temps)
print()

# --- Custom index (labels) ---
daily_temps = pd.Series(
    [22, 25, 28, 24, 30, 27, 23],
    index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    name="Temperature (°C)"
)
print("Custom string index:")
print(daily_temps)


In [ ]:
# --- Useful Series attributes ---
print("Name    :", daily_temps.name)
print("dtype   :", daily_temps.dtype)
print("Shape   :", daily_temps.shape)
print("Index   :", daily_temps.index.tolist())
print()

# --- Access by label ---
print("Wednesday:", daily_temps["Wed"])

# --- Access multiple labels ---
print("Weekend:")
print(daily_temps[["Sat", "Sun"]])


In [ ]:
# --- Vectorised math (no loop needed!) ---
print("Temperatures in Fahrenheit:")
fahrenheit = daily_temps * 9/5 + 32
print(fahrenheit.round(1))

print()
print("Days above 25°C:")
print(daily_temps[daily_temps > 25]) # filtering


## 1.2 — Creating DataFrames

A `DataFrame` is a **2-D labelled table** — rows + columns.  
You can think of it as a dict of Series that share the same index.


In [ ]:
# --- METHOD 1: From a dictionary (most common) ---
# Keys  → column names
# Values → lists (must all be same length)

data = {
    "Name":       ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "Age":        [25, 30, 35, 28, 22, 45],
    "City":       ["New York", "London", "Paris", "Tokyo", "Sydney", "Berlin"],
    "Salary":     [55000, 72000, 65000, 80000, 48000, 90000],
    "Experience": [2, 5, 8, 4, 1, 15],
    "Score":      [88, 92, 78, 95, 85, 70]
}

df = pd.DataFrame(data)
print("Our first DataFrame:")
df   # In Jupyter/Colab, just writing df renders a nice HTML table


In [ ]:
# --- METHOD 2: From a list of dicts ---
# Each dict = one row
rows = [
    {"product": "Laptop",  "price": 999,  "category": "Electronics"},
    {"product": "T-Shirt", "price": 25,   "category": "Clothing"},
    {"product": "Book",    "price": 15,   "category": "Education"},
]
products_df = pd.DataFrame(rows)
print("From list of dicts:")
print(products_df)


In [ ]:
# --- METHOD 3: From a CSV / Excel file ---
# pd.read_csv("your_file.csv")         # local file
# pd.read_csv("https://...")           # URL directly!
# pd.read_excel("your_file.xlsx")
# pd.read_json("your_file.json")

# We'll use a URL-hosted dataset later in Section 10.
# For now, let's use our dict-based df.
print("df is ready with shape:", df.shape)


## 1.3 — Inspecting a DataFrame

**These are the FIRST commands you always run on any new dataset.**


In [ ]:
# 1. Shape — (rows, columns)
print("Shape:", df.shape)
print(f"  → {df.shape[0]} rows, {df.shape[1]} columns")


In [ ]:
# 2. info() — column names, data types, non-null counts
# This is your most important diagnostic tool!
df.info()


In [ ]:
# 3. describe() — statistics for numeric columns
# count, mean, std, min, 25%, 50%, 75%, max
df.describe()


In [ ]:
# 4. head / tail / sample
print("=== First 3 rows ===")
print(df.head(3))
print()
print("=== Last 2 rows ===")
print(df.tail(2))
print()
print("=== Random 2 rows ===")
print(df.sample(2, random_state=42))


In [ ]:
# 5. Column names and dtypes
print("Columns:", df.columns.tolist())
print()
print("Data types:")
print(df.dtypes)


# Section 2 — Selection & Filtering

**This section is critical** — most of your day-to-day Pandas work is selecting
the right rows and columns.

There are 3 ways to select data:

| Method | Purpose | Syntax |
|--------|---------|--------|
| `df["col"]` or `df[["c1","c2"]]` | Select columns | Column name(s) |
| `.loc[rows, cols]` | Select by **label** | Index labels |
| `.iloc[rows, cols]` | Select by **position** | Integer position 0,1,2… |

---
## 2.1 — Column Selection


In [ ]:
# Single column → returns a Series
names = df["Name"]
print(type(names))   # <class 'pandas.core.series.Series'>
print(names)


In [ ]:
# Multiple columns → returns a DataFrame (note the double brackets)
subset = df[["Name", "Salary", "Score"]]
print(type(subset))  # <class 'pandas.core.frame.DataFrame'>
print(subset)


## 2.2 — Row Selection with `.loc` (label-based)

`.loc[row_labels, column_labels]`  
- Uses the **actual index labels** (often 0,1,2… by default, but can be strings)
- The end of a slice is **inclusive** (unlike Python)


In [ ]:
# Select a single row by label
print("Row with label 0:")
print(df.loc[0])


In [ ]:
# Select a range of rows (inclusive on both ends with .loc)
print("Rows 1 through 3:")
print(df.loc[1:3])


In [ ]:
# Select specific rows AND specific columns
print("Rows 0-2, columns Name and Salary:")
print(df.loc[0:2, ["Name", "Salary"]])


In [ ]:
# .loc is most powerful with a custom string index
# Let's set Name as the index to demonstrate
df_indexed = df.set_index("Name")
print("With Name as index:")
print(df_indexed)
print()
print("Look up Alice:")
print(df_indexed.loc["Alice"])
print()
print("Alice's salary:")
print(df_indexed.loc["Alice", "Salary"])


## 2.3 — Row Selection with `.iloc` (position-based)
`.iloc[row_positions, column_positions]`

- Uses integer positions — like a Python list (0-indexed)
- The end of a slice is exclusive (like Python)

In [ ]:
# First row (position 0)
print("First row:")
print(df.iloc[0])


In [ ]:
# Rows 0,1,2 — note: 3 is EXCLUDED (Python slicing)
print("First 3 rows:")
print(df.iloc[0:3])


In [ ]:
# Rows 0-2, columns 0 and 3 (Name and Salary)
# Column 0 = Name, Column 3 = Salary
print("First 3 rows, columns 0 and 3:")
print(df.iloc[0:3, [0, 3]])


In [ ]:
# Last 2 rows, all columns
print("Last 2 rows:")
print(df.iloc[-2:])


## 2.4 — Boolean Filtering (Masking)

This is the **most used** filtering technique in data science.  
You create a True/False mask, then use it to select rows.

```
condition → Series of True/False
df[condition] → rows where condition is True
```


In [ ]:
# Simple condition
mask = df["Salary"] > 70000
print("Boolean mask (True = salary > 70k):")
print(mask.values)

print()
print("Rows where salary > 70000:")
print(df[mask])


In [ ]:
# Combining conditions
# AND: use &   (not 'and')
# OR:  use |   (not 'or')
# NOT: use ~   (not 'not')
# ALWAYS wrap each condition in parentheses!

print("Age > 25 AND Score >= 85:")
print(df[(df["Age"] > 25) & (df["Score"] >= 85)])

print()
print("City is London OR Paris:")
print(df[(df["City"] == "London") | (df["City"] == "Paris")])


In [ ]:
# .isin() — cleaner than multiple OR conditions
cities = ["London", "Paris", "Tokyo"]
print(f"People in {cities}:")
print(df[df["City"].isin(cities)])


In [ ]:
# .between() — range filter (inclusive on both ends)
print("Age between 25 and 35:")
print(df[df["Age"].between(25, 35)])


In [ ]:
# .str.contains() — text search (supports regex!)
data_extra = pd.DataFrame({
    "name": ["Alice Johnson", "Bob Smith", "Charlie Brown", "Alice Wong"],
    "dept": ["Engineering", "Marketing", "Engineering", "HR"]
})
print("Names containing 'Alice':")
print(data_extra[data_extra["name"].str.contains("Alice")])


## 2.5 — The `.query()` Method

A cleaner, more readable way to filter — like writing SQL WHERE clauses.


In [ ]:
# Standard boolean filter (verbose)
result1 = df[(df["Age"] > 25) & (df["Score"] >= 85)]

# Same thing with .query() (much cleaner)
result2 = df.query("Age > 25 and Score >= 85")

print("Using .query():")
print(result2)

# You can also reference Python variables with @
min_salary = 70000
print()
print(f"Salary > {min_salary}:")
print(df.query("Salary > @min_salary"))
# The @ symbol lets you use a Python variable inside the query string.

# Section 3 — Data Cleaning

---
## Why This Matters

> Studies show data scientists spend **60–80% of their time cleaning data**.
>
> **Garbage in = Garbage out.** No ML model can compensate for dirty data.

Common problems:
- Extra whitespace: `"  Alice "` instead of `"Alice"`
- Inconsistent case: `"NEW YORK"` vs `"new york"` vs `"New York"`
- Wrong types: salary stored as `"$55,000"` (string) instead of `55000` (int)
- Invalid values: age of -5 or 200
- Duplicates: same row appears twice
- Bad dates: `"21/03/2021"` mixed with `"2021-03-21"`

---
## 3.1 — Building a Messy Dataset


In [ ]:
import pandas as pd
import numpy as np
import re

messy_data = {
    "name":       ["  Alice ", "BOB", "charlie", "Diana", "  Eve ", "Bob", "FRANK", "grace"],
    "age":        [25, 30, -5, 28, 200, 30, 45, 22],       # -5 and 200 are impossible
    "salary":     ["55000", "72000", "65000", None, "48000", "72000", "abc", "60000"],
    "city":       ["New York", "london", "PARIS", "Tokyo", "sydney", "London", "Berlin", "TOKYO"],
    "email":      ["alice@mail.com", "bob@mail.com", "charlie@mail.com", "diana@mail.com",
                   "eve@mail.com",   "bob@mail.com", "notanemail",       "grace@mail.com"],
    "join_date":  ["2020-01-15", "2019-06-20", "21/03/2021", "2020-09-10",
                   "2022-03-01", "2019-06-20", "2018-11-30", "2023-02-14"],
    "dept":       ["Engineering", "Marketing", "Engineering", None, "HR", "Marketing", "Finance", "hr"]
}

df_messy = pd.DataFrame(messy_data)
print("Raw messy data:")
print(df_messy.to_string())

In [ ]:
df_messy

## 3.2 — Cleaning Text: Strip, Case, Replace

In [ ]:
df = df_messy.copy()
# .str is the gateway to all text operations in Pandas.
# ── Strip whitespace ──────────────────────────────────────────────────────
# .str.strip() — removes spaces from the beginning and end of each text value.
df["name"] = df["name"].str.strip()
print("After strip:", df["name"].tolist())

# ── Standardise case ─────────────────────────────────────────────────────
df["name"] = df["name"].str.title()  # Title Case
df["city"] = df["city"].str.strip().str.title()
df["dept"] = df["dept"].str.strip().str.title()
print("After title case:")
print(df[["name", "city", "dept"]].to_string())


In [ ]:
# --- str methods cheat sheet ---
s = pd.Series(["  Hello World  ", "foo BAR", "  python  "])
print("Original    :", s.tolist())
print("strip       :", s.str.strip().tolist())
print("lower       :", s.str.lower().tolist())
print("upper       :", s.str.upper().tolist())
print("title       :", s.str.title().tolist())
print("replace     :", s.str.replace("Hello", "Hi", regex=False).tolist())
print("len         :", s.str.strip().str.len().tolist())
print("startswith  :", s.str.strip().str.startswith("H").tolist())


## 3.3 — Finding and Removing Duplicates

In [ ]:
print("=== Which rows are duplicates? ===")
print(df.duplicated())     # True = is a duplicate of a previous row

print()
print("=== Show the duplicate rows ===")
print(df[df.duplicated()])

print()
print(f"Total duplicates: {df.duplicated().sum()}")


In [ ]:

# df.drop_duplicates() ---> Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df)}")
print(df.to_string())


## 3.4 — Fixing Data Types

In [ ]:
print("Current dtypes:")
print(df.dtypes)
print()
print("Salary column values:", df["salary"].tolist())
errors="coerce" — if a value can't be converted (like "abc"), instead of crashing, it turns it into NaN (missing). This is very important — without it, one bad value crashes your whole operation.

# errors="coerce" — if a value can't be converted (like "abc"), instead of crashing, it turns it into NaN (missing). This is very important — without it, one bad value crashes your whole operation.
# 'None' → NaN, 'abc' → NaN when using errors='coerce'
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")
print()
print("After pd.to_numeric:")
print(df["salary"])


In [ ]:
# Fix mixed date formats
print("join_date BEFORE:", df["join_date"].tolist())
# pd.to_datetime() — converts text to proper date format.
# dayfirst=True — tells Pandas that in "21/03/2021", the 21 is the day (not the month). Important for non-US date formats.
df["join_date"] = pd.to_datetime(df["join_date"], dayfirst=True, errors="coerce")
print()
print("join_date AFTER:")
print(df["join_date"])
print("dtype:", df["join_date"].dtype)



## 3.5 — Handling Impossible / Outlier Values

In [ ]:
print("Age column:", df["age"].tolist())
print()
print("Ages outside [0, 120] are impossible — replace with NaN:")
df.loc[~df["age"].between(0, 120), "age"] = np.nan
print("Age column:", df["age"].tolist())


## 3.6 — Validating with Regex

In [ ]:
# Validate email addresses using a regular expression
EMAIL_PATTERN = r'^[\w.%+-]+@[\w.-]+\.[a-zA-Z]{2,}$'
# A regex (regular expression) is a pattern for matching text. This one describes what a valid email looks like (something@something.something).
df["email_valid"] = df["email"].str.match(EMAIL_PATTERN, na=False)
print(df[["name", "email", "email_valid"]].to_string())
# .str.match() — checks if each email matches the pattern. Returns True/False.
# na=False — if the email is missing (NaN), count it as invalid (False) instead of crashing.

## 3.7 — Cleaning Numeric Strings ($ , etc.)

In [ ]:
# price_df = pd.DataFrame({
#     "product": ["Laptop", "Phone", "Tablet"],
#     "price":   ["$1,299.00", "$799.50", "$449.00"]
# })
# print("Before cleaning:")
# print(price_df)

# # Remove $ and , then cast to float
# price_df["price_clean"] = (
#     price_df["price"]
#     .str.replace("$", "", regex=False)
#     .str.replace(",", "", regex=False)
#     .astype(float)
# )
# print()
# print("After cleaning:")
# print(price_df)


## 3.8 — Full Cleaning Pipeline

Always write cleaning as a reproducible pipeline — call it on any new batch of data.


In [ ]:
# def clean_employee_data(raw_df: pd.DataFrame) -> pd.DataFrame:
#     """Full cleaning pipeline for employee data."""
#     df = raw_df.copy()

#     # 1. Text: strip + title case
#     for col in ["name", "city", "dept"]:
#         df[col] = df[col].str.strip().str.title()

#     # 2. Numeric conversions
#     df["salary"] = pd.to_numeric(df["salary"], errors="coerce")

#     # 3. Dates
#     df["join_date"] = pd.to_datetime(df["join_date"], dayfirst=True, errors="coerce")

#     # 4. Impossible values → NaN
#     df.loc[~df["age"].between(0, 120), "age"] = np.nan

#     # 5. Duplicates
#     df = df.drop_duplicates().reset_index(drop=True)

#     return df


# df_clean = clean_employee_data(df_messy)
# print("Cleaned data:")
# print(df_clean.to_string())
# print()
# print("Missing values after cleaning:")
# print(df_clean.isnull().sum())


# Section 4 — Handling Missing Values

---
## 4.1 — Types of Missing Data

| Symbol | Meaning | When it appears |
|--------|---------|----------------|
| `NaN` | Not a Number | Numeric columns |
| `None` | Python None | Object columns |
| `NaT` | Not a Time | Datetime columns |
| `pd.NA` | Pandas NA | Newer nullable types |

All four are treated as "missing" by Pandas.

## 4.2 — Detecting Missing Values


In [ ]:
np.random.seed(42)
students = pd.DataFrame({
    "id":         range(1, 11),
    "name":       ["Alice", "Bob", None, "Diana", "Eve",
                   "Frank", "Grace", "Henry", "Iris", "Jack"],
    "math":       [85, np.nan, 78, 92, np.nan, 88, 76, np.nan, 95, 70],
    "english":    [90, 82, np.nan, 88, 75, np.nan, 80, 85, np.nan, 78],
    "science":    [88, 75, 80, np.nan, 82, 70, np.nan, 90, 85, np.nan],
    "attendance": [95, 88, 72, 98, np.nan, 85, np.nan, 91, 97, 80],
    "city":       ["NY", "LA", "NY", None, "Chicago", "LA", "NY", None, "Chicago", "LA"]
})
print(students.to_string())


In [ ]:
# --- Boolean mask ---
print("isnull() — True where missing:")
print(students.isnull())
#

# Creates a True/False table — True wherever there's a missing value. Like highlighting all the blank cells in Excel.

In [ ]:
# --- Count missing per column ---
# .sum() counts how many True values are in each column. Tells you exactly how many values are missing per column.
missing = pd.DataFrame({
    "count":   students.isnull().sum(),
    "percent": (students.isnull().sum() / len(students) * 100).round(1)
})
print("Missing values per column:")
print(missing[missing["count"] > 0])


In [ ]:
# --- Rows with at least one missing value ---
has_missing = students[students.isnull().any(axis=1)]
print(f"Rows with at least one missing value: {len(has_missing)}")
print(has_missing.to_string())
# any(axis=1) — checks each row: does it have at least one missing value? axis=1 means "check across columns." Returns the rows that have any NaN.

## 4.3 — Strategies Comparison

| Strategy | When to use |
|----------|------------|
| `dropna()` | Data is plentiful; missing rows are truly useless |
| Fill with **mean** | Numeric; roughly symmetric distribution |
| Fill with **median** | Numeric; outliers present (median is robust) |
| Fill with **mode** | Categorical |
| `ffill` / `bfill` | Time-series (use surrounding known values) |
| `interpolate()` | Smooth time-series (linear estimate between points) |
| Flag with a column | Keep the row; ML model can learn the missingness pattern |


In [ ]:
# ── STRATEGY 1: Drop ─────────────────────────────────────────────────────
# Removes every row that has even one missing value. Simple but often too aggressive — you might lose too much data.
df_drop = students.dropna()
print(f"After dropna: {len(df_drop)} rows remain (from {len(students)})")

# Drop only if specific columns are missing
df_drop2 = students.dropna(subset=["math", "english"])
print(f"After dropna(subset=['math','english']): {len(df_drop2)} rows")
# Only drop rows where math or english specifically is missing. Keeps rows where other columns are missing.


In [ ]:
# ── STRATEGY 2: Fill with mean / median ─────────────────────────────────
# .fillna() — fills all NaN values with something you choose.
# .median() — the middle value of the column. We use median instead of mean when there might be outliers (extreme values), because median isn't affected by them.
df_filled = students.copy()
num_cols = ["math", "english", "science", "attendance"]

for col in num_cols:
    median_val = df_filled[col].median()
    df_filled[col] = df_filled[col].fillna(median_val)
    print(f"Filled '{col}' NaN → median = {median_val:.1f}")

print()
print("Missing remaining:", df_filled[num_cols].isnull().sum().sum())


In [ ]:
# ── STRATEGY 3: Fill categorical with mode ──────────────────────────────
df_filled["city"] = df_filled["city"].fillna(df_filled["city"].mode()[0])
df_filled["name"] = df_filled["name"].fillna("Unknown")

print("City after fill:")
print(df_filled[["name", "city"]].to_string())
# For text columns, fill with the mode (most frequent value). You can't take an average of city names, but you can use the most common one.
# [0] — .mode() returns a list (in case of a tie), so we take the first one.

In [ ]:
# ── STRATEGY 4: Time-series strategies ──────────────────────────────────
ts = pd.DataFrame({
    "date":  pd.date_range("2024-01-01", periods=8, freq="D"),
    "price": [100, np.nan, np.nan, 105, np.nan, 110, np.nan, 115]
})

ts["ffill"]  = ts["price"].ffill()
ts["bfill"]  = ts["price"].bfill()
ts["interp"] = ts["price"].interpolate(method="linear")

print("Comparing strategies for time-series gaps:")
print(ts.to_string(index=False))

# ffill — "forward fill": if Monday's price is missing, use Friday's price (the last known value).
# bfill — "backward fill": if Monday's price is missing, use Tuesday's price (the next known value).
# interpolate(method="linear"): Smarter than ffill/bfill. If Monday = 100 and Wednesday = 106, it estimates Tuesday = 103 (halfway). Draws a straight line between known points.

In [ ]:
# # ── STRATEGY 5: Add a 'was_missing' flag ────────────────────────────────
# # Useful for ML — the model can learn that missingness itself is informative
# df_flag = students[["name", "math"]].copy()
# df_flag["math_was_missing"] = df_flag["math"].isnull().astype(int)
# df_flag["math"] = df_flag["math"].fillna(df_flag["math"].median())

# print("With missingness indicator:")
# print(df_flag.to_string())


# Section 5 — Transforming Data

- Transforming = converting existing data into a new, more useful form.
---
## 5.1 — `apply()` — Run a Function on Each Row or Column

`.apply()` is your Swiss Army knife for custom transformations.


In [ ]:
import pandas as pd
df = pd.DataFrame({
    "Name":   ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "Score":  [88, 92, 63, 95, 55],
    "Salary": [55000, 72000, 65000, 80000, 48000]
})

# --- Apply to a Series (each element) ---
def letter_grade(score):
    if score >= 90: return "A"
    elif score >= 80: return "B"
    elif score >= 70: return "C"
    elif score >= 60: return "D"
    else: return "F"

df["Grade"] = df["Score"].apply(letter_grade)
print("With grade column:")
print(df)

# .apply() runs that function on every single value in the column automatically. No loop needed.

In [ ]:
# # --- Lambda: shorter one-liners ---
# df["Salary_Monthly"] = df["Salary"].apply(lambda x: round(x / 12, 2))
# print()
# print("Monthly salary:")
# print(df[["Name", "Salary", "Salary_Monthly"]])


In [ ]:
# --- Apply across rows (axis=1) ---
def classify_employee(row):
    """Uses multiple columns at once."""
    if row["Score"] >= 90 and row["Salary"] >= 70000:
        return "High Performer"
    elif row["Score"] < 70:
        return "Needs Improvement"
    else:
        return "Average"

df["Category"] = df.apply(classify_employee, axis=1)
print()
print("Employee categories:")
print(df[["Name", "Score", "Salary", "Category"]])


## 5.2 — `map()` and `replace()` — Element-wise Lookup

Use `map` / `replace` when you have a lookup dictionary (much faster than `apply`).


In [ ]:
# map() — transform using a dictionary (one column)
city_region = {
    "New York": "East",
    "Los Angeles": "West",
    "Chicago": "Midwest",
    "Houston": "South"
}

city_df = pd.DataFrame({
    "city": ["New York", "Chicago", "Los Angeles", "New York", "Houston"]
})
city_df["region"] = city_df["city"].map(city_region)
print("City → Region mapping:")
print(city_df)


In [ ]:
# replace() — substitute specific values
raw = pd.DataFrame({"status": ["Y", "N", "Y", "N", "Y", "?"]})
raw["status_clean"] = raw["status"].replace({"Y": True, "N": False, "?": np.nan})
print("Status after replace:")
print(raw)


## 5.3 — `assign()` — Fluent Column Addition

`assign()` lets you add multiple columns in one chained expression — great for clean pipelines.


In [ ]:
result = (
    df
    .assign(
        Tax          = lambda d: (d["Salary"] * 0.20).round(2),
        Net_Salary   = lambda d: d["Salary"] - d["Tax"],
        Senior       = lambda d: d["Name"].apply(lambda n: len(n) > 4)  # silly example
    )
)
print(result[["Name", "Salary", "Tax", "Net_Salary", "Senior"]])


## 5.4 — `pd.cut()` and `pd.qcut()` — Binning Numbers

Convert continuous numbers into categorical bins (e.g., "Low / Medium / High").


In [ ]:
salaries = pd.Series([30000, 45000, 55000, 70000, 85000, 100000, 130000, 200000])
# pd.cut() — Turn Numbers into Categoriess
# cut() — you define the bin edges
labels = ["Entry", "Junior", "Mid", "Senior", "Lead"]
bins   = [0, 40000, 60000, 80000, 100000, float("inf")]

salary_bands = pd.cut(salaries, bins=bins, labels=labels)
print("pd.cut() — fixed boundaries:")
print(pd.DataFrame({"salary": salaries, "band": salary_bands}))


In [ ]:
# qcut() — automatically makes equal-sized groups (by quantile)
salary_quartile = pd.qcut(salaries, q=4, labels=["Q1 (Bottom)", "Q2", "Q3", "Q4 (Top)"])
print("\npd.qcut() — equal-size quartile groups:")
print(pd.DataFrame({"salary": salaries, "quartile": salary_quartile}))


## 5.5 — `get_dummies()` — One-Hot Encoding

Machine learning models need **numbers**, not text categories.  
One-hot encoding converts `"Engineering"` → three binary columns.

```
dept                    dept_Engineering  dept_HR  dept_Marketing
Engineering    →              1              0          0
HR             →              0              1          0
Marketing      →              0              0          1
```


In [ ]:
emp = pd.DataFrame({
    "name":   ["Alice", "Bob", "Charlie", "Diana"],
    "salary": [55000, 72000, 65000, 80000],
    "dept":   ["Engineering", "Marketing", "Engineering", "HR"]
})

encoded = pd.get_dummies(emp, columns=["dept"], drop_first=False)
print("After one-hot encoding:")
print(encoded)
# drop_first=True removes one column to avoid multicollinearity in regression


# Section 6 — DateTime Operations

Date/time data is everywhere: sales timestamps, sensor logs, stock prices.  
Pandas has first-class support for it.

---



## 6.1 — Parsing Dates

In [ ]:
# pd.to_datetime() is very flexible
dates = ["2024-01-15", "15/03/2024", "March 20 2024", "20240401"]
parsed = pd.to_datetime(dates, infer_datetime_format=True, errors="coerce")
print("Parsed dates:")
for orig, parsed_dt in zip(dates, parsed):
    print(f"  {orig!r:20s} → {parsed_dt}")


In [ ]:
# Create a realistic time-series dataset
np.random.seed(0)
n = 200
dates = pd.date_range(start="2023-01-01", periods=n, freq="D")

sales_df = pd.DataFrame({
    "date":     dates,
    "revenue":  np.random.normal(loc=1000, scale=200, size=n).clip(200),
    "orders":   np.random.randint(10, 80, size=n),
    "category": np.random.choice(["Electronics", "Clothing", "Food"], n)
})
sales_df["date"] = pd.to_datetime(sales_df["date"])
print("Sample sales data:")
print(sales_df.head(8))


## 6.2 — Extracting Date Components

In [ ]:
# Access year, month, day, weekday, hour, etc. via .dt accessor
sales_df["year"]     = sales_df["date"].dt.year
sales_df["month"]    = sales_df["date"].dt.month
sales_df["month_name"] = sales_df["date"].dt.strftime("%B")  # "January"
sales_df["weekday"]  = sales_df["date"].dt.day_name()
sales_df["is_weekend"] = sales_df["date"].dt.dayofweek >= 5  # Mon=0, Sun=6
sales_df["quarter"]  = sales_df["date"].dt.quarter

print(sales_df[["date", "year", "month", "month_name", "weekday", "is_weekend", "quarter"]].head(10))


## 6.3 — Resampling (Group by Time Period)

`resample()` is like `groupby()` but for time periods.  
Set the date as the index first.


In [ ]:
ts = sales_df.set_index("date")

# Weekly total revenue
weekly = ts["revenue"].resample("W").sum().round(2)
print("Weekly total revenue (first 6 weeks):")
print(weekly.head(6))

print()

# Monthly average orders
monthly = ts["orders"].resample("ME").agg(["mean", "sum"]).round(1)
monthly.columns = ["avg_orders", "total_orders"]
print("Monthly order stats:")
print(monthly)


## 6.4 — Rolling Windows (Moving Averages)

Used everywhere in finance and signal processing — smooth out noise.


In [ ]:
# 7-day rolling average of revenue
ts["revenue_7d_avg"] = ts["revenue"].rolling(window=7).mean().round(1)

# 30-day rolling max
ts["revenue_30d_max"] = ts["revenue"].rolling(window=30).max().round(1)

print("Revenue with rolling averages:")
print(ts[["revenue", "revenue_7d_avg", "revenue_30d_max"]].dropna().head(10))


In [ ]:
# Rolling std — how volatile is revenue?
ts["revenue_std7"] = ts["revenue"].rolling(7).std().round(1)
print("Revenue 7-day rolling std (volatility):")
print(ts[["revenue", "revenue_7d_avg", "revenue_std7"]].dropna().head(10))


# Section 7 — Merging & Joining DataFrames

In the real world, data lives in multiple tables.  
You need to **join** them — exactly like SQL.

---

## 7.1 — The Four Types of Join

```
LEFT TABLE    RIGHT TABLE
  A B           B C
  1 x           x p     INNER  → only rows where B matches in BOTH
  2 y           z q     LEFT   → all left rows, NaN if no right match
  3 y                   RIGHT  → all right rows, NaN if no left match
                        OUTER  → all rows from both, NaN where no match
```

## 7.2 — Setup


In [ ]:
employees = pd.DataFrame({
    "emp_id":    [101, 102, 103, 104, 105, 106],
    "name":      ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "dept_id":   [1, 2, 1, 3, 2, 4]        # dept 4 has no name; Frank→dept 4
})

departments = pd.DataFrame({
    "dept_id":   [1, 2, 3, 5],             # dept 5 has no employee; dept 4 missing
    "dept_name": ["Engineering", "Marketing", "HR", "Finance"],
    "location":  ["NYC", "LA", "Chicago", "Boston"]
})

salaries = pd.DataFrame({
    "emp_id":  [101, 102, 103, 104, 107],  # 107 not in employees; 105/106 missing
    "salary":  [85000, 72000, 90000, 68000, 55000],
    "bonus":   [5000, 3000, 8000, 2000, 1000]
})

print("Employees:"); print(employees)
print("\nDepartments:"); print(departments)
print("\nSalaries:"); print(salaries)


In [ ]:
# ── INNER JOIN: only matching rows ──────────────────────────────────────
inner = pd.merge(employees, departments, on="dept_id", how="inner")
print("INNER JOIN — only employees with a known department:")
print(inner)
print(f"  {len(inner)} rows (Frank loses dept_id=4 has no name)")


In [ ]:
# ── LEFT JOIN: all employees, dept info where available ─────────────────
left = pd.merge(employees, departments, on="dept_id", how="left")
print("LEFT JOIN — all employees kept:")
print(left)
print(f"  {len(left)} rows (Frank's dept columns are NaN)")


In [ ]:
# ── RIGHT JOIN: all departments ────────────────────────────────────────
right = pd.merge(employees, departments, on="dept_id", how="right")
print("RIGHT JOIN — all departments kept:")
print(right)
print(f"  {len(right)} rows (Finance dept kept even with no employees)")


In [ ]:
# ── OUTER JOIN: everything from both ───────────────────────────────────
outer = pd.merge(employees, departments, on="dept_id", how="outer")
print("OUTER JOIN — all rows from both:")
print(outer)


In [ ]:
# ── Chaining multiple merges ─────────────────────────────────────────────
full = (
    employees
    .merge(departments, on="dept_id", how="left")
    .merge(salaries,   on="emp_id",   how="left")
)
print("Full employee data (chained merges):")
print(full.to_string())


## 7.3 — `pd.concat()` — Stacking DataFrames

Use `concat` to **add rows** (not join by key).  
Common use case: data from multiple months, files, or sources.


In [ ]:
jan = pd.DataFrame({"month": "Jan", "product": ["A","B","C"], "sales": [100,200,150]})
feb = pd.DataFrame({"month": "Feb", "product": ["A","B","D"], "sales": [120,180,90]})
mar = pd.DataFrame({"month": "Mar", "product": ["B","C","D"], "sales": [210,160,130]})

all_months = pd.concat([jan, feb, mar], ignore_index=True)
print("Stacked sales data:")
print(all_months)



# Section 8 — GroupBy & Aggregation

---
## The split-apply-combine pattern

```
           SPLIT                  APPLY          COMBINE
df  →  group by "dept"  →  mean(salary)  →  one row per dept
```

Think of it as Excel's **Pivot Tables**, but on steroids.

---


## 8.1 — Basic GroupBy


In [ ]:
np.random.seed(42)
company = pd.DataFrame({
    "name":        [f"Emp_{i}" for i in range(1, 21)],
    "dept":        np.random.choice(["Engineering","Marketing","HR","Finance","Sales"], 20),
    "gender":      np.random.choice(["M","F"], 20),
    "salary":      np.random.randint(45000, 120000, 20),
    "years_exp":   np.random.randint(1, 15, 20),
    "performance": np.random.choice(["Poor","Average","Good","Excellent"], 20)
})
print("Employee data:")
print(company.to_string(index=False))


In [ ]:
# Average salary by department
avg_sal = company.groupby("dept")["salary"].mean().round(0)
print("Average salary by department:")
print(avg_sal.sort_values(ascending=False))


In [ ]:
# Multiple aggregation functions at once
stats = (
    company
    .groupby("dept")["salary"]
    .agg(["mean", "median", "min", "max", "count"])
    .round(0)
    .rename(columns={"count": "headcount"})
)
print("Salary stats by department:")
print(stats)


## 8.2 — Multi-Column GroupBy

In [ ]:
# Group by two columns
by_dept_gender = (
    company
    .groupby(["dept", "gender"])["salary"]
    .mean()
    .round(0)
)
print("Avg salary by dept + gender:")
print(by_dept_gender)
print()

# Unstack gender into columns (nice table view)
print("Unstacked (dept = rows, gender = columns):")
print(by_dept_gender.unstack(fill_value=0))


## 8.3 — Custom Aggregation with `.agg()`

In [ ]:
# Different functions for different columns
custom = company.groupby("dept").agg(
    avg_salary  = ("salary",    "mean"),
    max_salary  = ("salary",    "max"),
    total_exp   = ("years_exp", "sum"),
    headcount   = ("name",      "count")
).round(1)

print("Custom aggregation:")
print(custom)


## 8.4 — `transform()` — Add Group Stats Back to Original Rows

`groupby().agg()` → returns a shorter DataFrame (one row per group)  
`groupby().transform()` → returns the **same shape** as the original, one value per row

This is essential for feature engineering in ML!


In [ ]:
# Add department average salary to each employee's row
company["dept_avg_salary"] = (
    company.groupby("dept")["salary"].transform("mean").round(0)
)

# How much does each person earn vs their dept average?
company["vs_dept_avg"] = (company["salary"] - company["dept_avg_salary"]).round(0)

print("Each employee vs their department average:")
cols = ["name", "dept", "salary", "dept_avg_salary", "vs_dept_avg"]
print(company[cols].sort_values("dept").to_string(index=False))


## 8.5 — `filter()` — Keep Only Groups Meeting a Condition

In [ ]:
# Keep only employees from departments with avg salary > $70k
high_pay_depts = company.groupby("dept").filter(
    lambda x: x["salary"].mean() > 70000
)
print("Employees in high-paying departments (avg > $70k):")
print(high_pay_depts[["name","dept","salary"]].sort_values("dept").to_string(index=False))


## 8.6 — Pivot Tables

In [ ]:
pivot = company.pivot_table(
    values  = "salary",
    index   = "dept",
    columns = "performance",
    aggfunc = "mean",
    fill_value = 0
).round(0)

print("Pivot table — avg salary by dept & performance:")
print(pivot)


# Section 9 — Performance & Best Practices

---


## 9.1 — Use Efficient Data Types

The default dtypes Pandas infers are often wasteful.  
Casting to the right dtype can **reduce memory 50–90%**.




In [ ]:
# Build a large-ish DataFrame to benchmark
n = 100_000
big_df = pd.DataFrame({
    "id":       np.random.randint(0, 10000, n),
    "category": np.random.choice(["A","B","C","D","E"], n),   # low-cardinality string
    "score":    np.random.uniform(0, 100, n),
    "flag":     np.random.choice([True, False], n),
    "count":    np.random.randint(0, 255, n),                  # fits in uint8
})

print(f"Before optimisation — memory: {big_df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(big_df.dtypes)


In [ ]:
# Optimise dtypes
big_df_opt = big_df.copy()

# category — for low-cardinality string columns (huge savings)
big_df_opt["category"] = big_df_opt["category"].astype("category")

# float32 instead of float64 (half the memory, usually enough precision)
big_df_opt["score"] = big_df_opt["score"].astype("float32")

# uint8 for small integers (0-255 fits in 1 byte instead of 8)
big_df_opt["count"] = big_df_opt["count"].astype("uint8")

print(f"After optimisation — memory: {big_df_opt.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(big_df_opt.dtypes)


## 9.2 — Vectorise: Avoid Python Loops

The cardinal sin in Pandas is using a Python loop when a vectorised operation exists.


In [ ]:
import time

df_perf = pd.DataFrame({"a": np.random.randn(100_000), "b": np.random.randn(100_000)})

# Slow: Python loop
start = time.time()
result_loop = []
for i in range(len(df_perf)):
    result_loop.append(df_perf["a"].iloc[i] + df_perf["b"].iloc[i])
loop_time = time.time() - start

# Fast: vectorised
start = time.time()
result_vec = df_perf["a"] + df_perf["b"]
vec_time = time.time() - start

print(f"Loop time   : {loop_time:.3f}s")
print(f"Vectorised  : {vec_time:.4f}s")
print(f"Speedup     : {loop_time / max(vec_time, 0.0001):.0f}x faster")


## 9.3 — Method Chaining

Write transformations as a readable chain using `.pipe()` and `assign()`.


In [ ]:
raw = pd.DataFrame({
    "Name":   ["  alice ", "BOB", "charlie "],
    "Salary": ["55,000", "72,000", "65,000"],
    "Dept":   ["engineering", "MARKETING", "hr"]
})

def parse_salary(df):
    df = df.copy()
    df["Salary"] = df["Salary"].str.replace(",", "").astype(int)
    return df

clean = (
    raw
    .assign(
        Name   = lambda d: d["Name"].str.strip().str.title(),
        Dept   = lambda d: d["Dept"].str.strip().str.title()
    )
    .pipe(parse_salary)
    .assign(Monthly = lambda d: (d["Salary"] / 12).round(2))
    .sort_values("Salary", ascending=False)
    .reset_index(drop=True)
)
print("Chained pipeline result:")
print(clean)


## 9.4 — Common Gotchas

```python
# SettingWithCopyWarning — modifying a slice
subset = df[df["age"] > 25]
subset["new_col"] = 1   # Warning! May or may not modify df

# Use .copy() to be explicit
subset = df[df["age"] > 25].copy()
subset["new_col"] = 1   # Safe

# Comparing floats with ==
df[df["score"] == 0.1]  # May miss due to floating-point precision

# Use np.isclose for floats
df[np.isclose(df["score"], 0.1)]
```


# Section 10 — Real-World Mini Project

## E-Commerce Sales Analysis Pipeline

We'll simulate a realistic dataset and run a complete analysis pipeline:

1. Generate raw, messy data  
2. Clean it  
3. Handle missing values  
4. Feature engineer  
5. Aggregate & analyse  
6. Produce a summary report  

---



## 10.1 — Generate Raw Data

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(99)
N = 500

# Simulate raw orders table (as you'd receive from a database export)
orders_raw = pd.DataFrame({
    "order_id":   range(1001, 1001 + N),
    "customer_id": np.random.randint(200, 350, N),
    "order_date":  pd.date_range("2023-01-01", periods=N, freq="h").astype(str),
    "product":     np.random.choice(
                       ["Laptop", "Phone", "Tablet", "Headphones",
                        "Charger", "keyboard", "MOUSE ", "monitor"],  # messy case
                       N),
    "category":    np.random.choice(["Electronics", "Accessories", None], N,
                       p=[0.5, 0.4, 0.1]),
    "price":       np.random.choice(
                       [999, 699, 499, 149, 29, 79, 25, 399, "N/A", None], N),
    "quantity":    np.random.choice([1, 1, 1, 2, 3, -1, 0, None], N,
                       p=[0.4, 0.2, 0.15, 0.1, 0.07, 0.03, 0.03, 0.02]),
    "region":      np.random.choice(["North", "South", "East", "West", "NORTH", "east"], N),
    "status":      np.random.choice(["completed", "Completed", "CANCELLED", "pending", None], N,
                       p=[0.5, 0.2, 0.15, 0.1, 0.05])
})

print(f"Raw orders: {orders_raw.shape}")
print(orders_raw.head(10).to_string())


In [ ]:
# Quick quality check
print("=== DATA QUALITY REPORT ===")
print()
print("Missing values:")
print(orders_raw.isnull().sum())
print()
print("Unique products (before cleaning):", orders_raw["product"].unique())
print()
print("Unique regions (before cleaning):", orders_raw["region"].unique())
print()
print("Unique statuses (before cleaning):", orders_raw["status"].unique())


## 10.2 — Cleaning Pipeline

In [ ]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    """Full cleaning pipeline for the orders dataset."""
    d = df.copy()

    # ── Text columns ──────────────────────────────────────────────────────
    d["product"]  = d["product"].str.strip().str.title()
    d["region"]   = d["region"].str.strip().str.title()
    d["status"]   = d["status"].str.strip().str.lower()

    # ── Standardise status values ─────────────────────────────────────────
    d["status"] = d["status"].replace({"completed": "completed"})  # already normalised

    # ── Dates ─────────────────────────────────────────────────────────────
    d["order_date"] = pd.to_datetime(d["order_date"], errors="coerce")

    # ── Price: "N/A" → NaN, then cast to float ────────────────────────────
    d["price"] = pd.to_numeric(d["price"], errors="coerce")

    # ── Quantity: negative/zero → NaN ────────────────────────────────────
    d["quantity"] = pd.to_numeric(d["quantity"], errors="coerce")
    d.loc[d["quantity"] <= 0, "quantity"] = np.nan

    # ── Fill missing category with mode ───────────────────────────────────
    d["category"] = d["category"].fillna(d["category"].mode()[0])

    # ── Fill missing price with median per product ────────────────────────
    d["price"] = d.groupby("product")["price"].transform(
        lambda x: x.fillna(x.median())
    )

    # ── Fill missing quantity with 1 (most common) ────────────────────────
    d["quantity"] = d["quantity"].fillna(1).astype(int)

    # ── Drop rows with no status ──────────────────────────────────────────
    d = d.dropna(subset=["status"])

    # ── Remove duplicates ─────────────────────────────────────────────────
    d = d.drop_duplicates(subset=["order_id"])

    return d.reset_index(drop=True)


orders = clean_orders(orders_raw)
print(f"Cleaned orders: {orders.shape}")
print()
print("Missing values after cleaning:")
print(orders.isnull().sum())


## 10.3 — Feature Engineering

In [ ]:
# Derive new useful columns
orders = orders.assign(
    revenue     = lambda d: d["price"] * d["quantity"],
    year        = lambda d: d["order_date"].dt.year,
    month       = lambda d: d["order_date"].dt.month,
    month_name  = lambda d: d["order_date"].dt.strftime("%b"),
    day_of_week = lambda d: d["order_date"].dt.day_name(),
    is_weekend  = lambda d: d["order_date"].dt.dayofweek >= 5,
    is_cancelled= lambda d: (d["status"] == "CANCELLED").astype(int),
    price_tier  = lambda d: pd.cut(
        d["price"],
        bins=[0, 50, 200, 700, float("inf")],
        labels=["Budget", "Mid-range", "Premium", "Luxury"]
    )
)

print("Engineered features added:")
print(orders[["order_id","product","price","quantity","revenue","price_tier","is_weekend"]].head(10))


## 10.4 — Analysis

In [ ]:
# ── Revenue by product ───────────────────────────────────────────────────
print("=== Top Products by Revenue ===")
prod_revenue = (
    orders.groupby("product")
    .agg(
        orders    = ("order_id", "count"),
        revenue   = ("revenue",  "sum"),
        avg_price = ("price",    "mean")
    )
    .round(2)
    .sort_values("revenue", ascending=False)
)
print(prod_revenue)


In [ ]:
# ── Monthly revenue trend ────────────────────────────────────────────────
print("\n=== Monthly Revenue ===")
monthly_rev = (
    orders[orders["status"] != "cancelled"]
    .groupby(["year", "month_name", "month"])["revenue"]
    .sum()
    .reset_index()
    .sort_values("month")
    .drop("month", axis=1)
)
monthly_rev.columns = ["Year", "Month", "Revenue"]
print(monthly_rev.to_string(index=False))


In [ ]:
# ── Regional analysis ────────────────────────────────────────────────────
print("\n=== Revenue & Cancellation Rate by Region ===")
regional = orders.groupby("region").agg(
    total_revenue    = ("revenue",      "sum"),
    total_orders     = ("order_id",     "count"),
    cancelled_orders = ("is_cancelled", "sum"),
).assign(
    avg_order_value   = lambda d: (d["total_revenue"] / d["total_orders"]).round(2),
    cancellation_rate = lambda d: (d["cancelled_orders"] / d["total_orders"] * 100).round(1)
).round(2)
print(regional.to_string())


In [ ]:
# ── Weekend vs Weekday ────────────────────────────────────────────────────
print("\n=== Weekend vs Weekday Orders ===")
wk_vs_wkd = orders.groupby("is_weekend").agg(
    orders  = ("order_id", "count"),
    revenue = ("revenue",  "sum"),
    avg_rev = ("revenue",  "mean")
).round(2)
wk_vs_wkd.index = wk_vs_wkd.index.map({False: "Weekday", True: "Weekend"})
print(wk_vs_wkd)


In [ ]:
# ── Customer analysis ─────────────────────────────────────────────────────
print("\n=== Top 10 Customers by Revenue ===")
customer_summary = (
    orders.groupby("customer_id")
    .agg(
        total_revenue = ("revenue",  "sum"),
        order_count   = ("order_id", "count"),
        avg_order     = ("revenue",  "mean"),
    )
    .round(2)
    .sort_values("total_revenue", ascending=False)
    .head(10)
)
print(customer_summary.to_string())


## 10.5 — Final Summary Report

In [ ]:
total_rev    = orders[orders["status"] != "cancelled"]["revenue"].sum()
total_orders = len(orders)
cancel_rate  = (orders["status"] == "cancelled").mean() * 100
avg_order    = orders["revenue"].mean()
top_product  = orders.groupby("product")["revenue"].sum().idxmax()
top_region   = orders.groupby("region")["revenue"].sum().idxmax()

print("=" * 50)
print("E-COMMERCE PERFORMANCE SUMMARY")
print("=" * 50)
print(f"  Total Revenue      : ${total_rev:>12,.2f}")
print(f"  Total Orders       : {total_orders:>12,}")
print(f"  Avg Order Value    : ${avg_order:>12,.2f}")
print(f"  Cancellation Rate  : {cancel_rate:>11.1f}%")
print(f"  Top Product        : {top_product}")
print(f"  Top Region         : {top_region}")
print("=" * 50)
